# 第6课：综合实战 — 端到端训练

**学习目标：**
- 完整的 PyTorch 训练流程（数据 → 模型 → 训练 → 评估）
- 可视化决策边界
- 训练过程的 loss 曲线
- 对比 NumPy 和 PyTorch 的效果

---

本课是 PyTorch 教程的综合实战。我们将完成一个完整的分类项目，并可视化决策边界的形成过程。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from pytorch.utils import create_data, plot_data

## 6.1 数据准备

In [ ]:
# 生成数据
train_data = create_data(800)
test_data = create_data(200)

# 转为 Tensor
X_train = torch.tensor(train_data[:, :2], dtype=torch.float32)
y_train = torch.tensor(train_data[:, 2], dtype=torch.long)
X_test = torch.tensor(test_data[:, :2], dtype=torch.float32)
y_test = torch.tensor(test_data[:, 2], dtype=torch.long)

print(f"训练集: {X_train.shape[0]} 样本")
print(f"测试集: {X_test.shape[0]} 样本")

plt.figure(figsize=(6, 5))
plot_data(train_data, "训练数据")
plt.show()

## 6.2 模型定义

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net().to(device)
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"设备: {device}")
print(f"参数量: {sum(p.numel() for p in model.parameters())}")

## 6.3 训练

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

train_losses = []
test_losses = []
n_epochs = 300

for epoch in range(n_epochs):
    # ---- 训练 ----
    model.train()
    logits = model(X_train)
    train_loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
    train_losses.append(train_loss.item())
    
    # ---- 验证 ----
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test)
        test_loss = criterion(test_logits, y_test)
        test_losses.append(test_loss.item())
    
    if (epoch + 1) % 50 == 0:
        # 计算准确率
        preds = torch.argmax(test_logits, dim=1)
        acc = (preds == y_test).float().mean()
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss.item():.4f} | Test Loss: {test_loss.item():.4f} | Acc: {acc:.2%}")

## 6.4 可视化

In [ ]:
plt.figure(figsize=(14, 5))

# Loss 曲线
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train')
plt.plot(test_losses, label='Test')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()

# 训练数据的真实分布
plt.subplot(1, 3, 2)
plot_data(train_data, "Ground Truth")

# 模型预测
model.eval()
with torch.no_grad():
    preds = torch.argmax(model(X_test), dim=1).cpu().numpy()
test_result = test_data.copy()
test_result[:, 2] = preds
plt.subplot(1, 3, 3)
plot_data(test_result, "Predictions")

plt.tight_layout()
plt.show()

## 6.5 决策边界可视化

在二维平面上绘制模型的决策边界，直观展示网络学到了什么：

In [ ]:
# 生成网格点
xx, yy = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32).to(device)

# 预测
model.eval()
with torch.no_grad():
    Z = torch.argmax(model(grid), dim=1).cpu().numpy()
Z = Z.reshape(xx.shape)

# 绘制
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
plt.scatter(test_data[:, 0], test_data[:, 1], c=test_data[:, 2], s=5, cmap='RdYlBu', edgecolors='k', linewidths=0.3)
plt.title('Decision Boundary')
plt.xlabel('x')
plt.ylabel('y')
plt.colorbar()
plt.show()

## 6.6 保存和加载模型

In [ ]:
# 保存模型
torch.save(model.state_dict(), 'model_weights.pth')
print("模型已保存")

# 加载模型
loaded_model = Net().to(device)
loaded_model.load_state_dict(torch.load('model_weights.pth'))
loaded_model.eval()
print("模型已加载")

# 验证加载的模型
with torch.no_grad():
    orig_preds = torch.argmax(model(X_test[:5]), dim=1)
    load_preds = torch.argmax(loaded_model(X_test[:5]), dim=1)
    print(f"原始预测: {orig_preds}")
    print(f"加载预测: {load_preds}")
    print(f"一致: {torch.equal(orig_preds, load_preds)}")

---

## 总结：NumPy vs PyTorch 全面对比

| 方面 | NumPy 手写 | PyTorch |
| --- | --- | --- |
| **网络构建** | 手写 Layer/Network 类 | nn.Module + nn.Sequential |
| **前向传播** | 手动矩阵运算 | `model(x)` 一行 |
| **损失函数** | 自定义 1-dot 损失 | nn.CrossEntropyLoss |
| **反向传播** | 手动链式法则 | loss.backward() 自动 |
| **权重更新** | `W += lr * grad` | optimizer.step() |
| **GPU 加速** | 不支持 | `.to(device)` |
| **代码量** | ~200 行 | ~50 行 |
| **灵活性** | 完全可控 | 框架封装 |

### 学习路线回顾

```
NumPy 教程                        PyTorch 教程
────────────────────              ────────────────────
01 NumPy 基础          ←→         01 Tensor 基础
02 神经元              ←→         03 nn.Module
03 Layer/Network       ←→         03 nn.Module
04 Softmax             ←→         04 Softmax 分类
05 分类任务            ←→         04 Softmax 分类
06 损失+需求函数       ←→         05 损失+优化器
07 反向传播训练        ←→         05 损失+优化器
08 自动微分            ←→         02 Autograd
```

---

恭喜你完成了全部教程！你已经理解了神经网络从底层原理到框架使用的完整链路。